In [0]:
import time
from datetime import datetime
import random
import json
import os

def ejecutar_simulador_con_fallas_reales(ciclos_totales=30):
    print("🚀 [SCADA] Iniciando generación de archivos JSON con FALLAS REALES DE CAMPO...")
    print("---------------------------------------------------------------------")
    
    ruta_local = "/Workspace/Shared/telemetria_raw/"
    os.makedirs(ruta_local, exist_ok=True)
    
    presion_nominal = 1200.0
    caudal_nominal = 1500.0
    
    for i in range(ciclos_totales):
        ahora = datetime.now()
        timestamp_str = ahora.strftime("%Y-%m-%d %H:%M:%S")
        
        # --- ACÁ METEMOS LOS ERRORES DE LA REALIDAD ---
        if i == 5:
            # Falla 1: El sensor se desconecta y manda un campo vacío (Nulo)
            presion_raw = ""
            caudal_raw = str(round(caudal_nominal + random.uniform(-10, 10), 2))
            print("⚠️ [SIMULADOR] Inyectando error: Campo de presión vacío (Nulo).")
        
        elif i == 10:
            # Falla 2: Interferencia eléctrica en la antena, manda texto corrupto
            presion_raw = "ERR_VAL_99"
            caudal_raw = "0.0"
            print("⚠️ [SIMULADOR] Inyectando error: Texto corrupto por ruido en la red.")
            
        elif i == 12:
            # Falla 3: El descalibrador tira un valor negativo físicamente imposible
            presion_raw = "-999.0"
            caudal_raw = str(round(caudal_nominal, 2))
            print("⚠️ [SIMULADOR] Inyectando error: Presión negativa imposible.")
            
        else:
            # Comportamiento normal o caída operativa (Vuelta 15 en adelante)
            if i >= 15:
                presion = presion_nominal - 450.0 + random.uniform(-5, 5)
                caudal = caudal_nominal - 850.0 + random.uniform(-20, 20)
            else:
                presion = presion_nominal + random.uniform(-10, 10)
                caudal = caudal_nominal + random.uniform(-20, 20)
            
            presion_raw = str(round(presion, 2))
            caudal_raw = str(round(caudal, 2))

        # Armamos el archivo con la basura incluida
        diccionario_sensor = {
            "id_pozo": "POZO_MEDALLION_01",
            "timestamp_sensor": timestamp_str,
            "presion_cabezal_raw": presion_raw,
            "caudal_bpd_raw": caudal_raw
        }
        
        nombre_archivo = f"lectura_pozo_{ahora.strftime('%Y%m%d_%H%M%S')}_{i}.json"
        ruta_archivo_completa = os.path.join(ruta_local, nombre_archivo)
        
        with open(ruta_archivo_completa, "w") as archivo_json:
            json.dump(diccionario_sensor, archivo_json)
            
        time.sleep(2) 
        
    print("---------------------------------------------------------------------")
    print("✅ Simulación con fallas terminada.")

ejecutar_simulador_con_fallas_reales(ciclos_totales=30)